# A Multimodal Supervisor/Planner/Worker Agent (after Jockey)

https://www.langchain.com/blog/jockey-twelvelabs-langgraph describes Jockey,
a conversational video agent built from a **Supervisor** (routes and
coordinates), a **Planner** (breaks complex requests into steps), and
specialized **Workers** (video search / text generation / editing).

psychscanner has no video pipeline, but the same three-role shape
generalizes to whatever content blocks a trial's stimulus carries (image /
audio / text — see `psychscanner.datasets.prompts.multimodal`): a
single-block-type stimulus routes straight to the matching worker; a
mixed-block stimulus goes through the planner, which dispatches each worker
on its slice of the content, then an aggregate step combines their findings.
Nothing like this exists in psychscanner today, so
`psychscanner.agents.make_supervisor_agent` builds it directly on LangGraph.

This notebook is about the routing/dispatch *mechanics*, so it runs against
`mock-llm` (fast, deterministic, no local model download) — each worker's
model call is real, it just echoes back what it received; swap in any real
chat model (as in the earlier notebooks) for genuine image/audio
understanding, the graph structure doesn't change.

In [1]:
from pathlib import Path

from langchain_core.messages import HumanMessage

import psychscanner as psy
from psychscanner.agents import make_supervisor_agent
from psychscanner.memories import ChatMockModel
from psychscanner.task_runner import TaskRunner

RUN_DIR = Path.cwd() / "_supervisor_tutorial_run"
RUN_DIR.mkdir(exist_ok=True)

import base64

png_bytes = base64.b16decode(
    "89504E470D0A1A0A0000000D49484452000000010000000108020000009077"
    "53DE0000000C4944415478DA6360000002000155A1F6B50000000049454E44AE426082"
)
(RUN_DIR / "pixel.png").write_bytes(png_bytes)

# A bigger echo buffer than the default 10 chars, so the aggregate step's
# echo doesn't re-truncate the concatenated worker outputs into illegibility.
model = ChatMockModel(model="mock-chat-model", repeat_buffer_length=200)
agent = make_supervisor_agent(model)
print("PsychScanner successfully imported!")

PsychScanner successfully imported!


## 1. Single block type — routes straight to one worker

An image-only stimulus skips the planner entirely: the supervisor routes it
directly to `vision_worker`.

In [2]:
from psychscanner.datasets.prompts.multimodal import image_block

tasktrials = {"trials": [
    {"trcode": "t1", "stimulus": HumanMessage(content=[image_block(RUN_DIR / "pixel.png")]),
     "tasktype": "x", "parser": None, "fb": False},
]}
runner = TaskRunner(
    scanning_agent=agent, trace_cfg={"trial": "tut-", "task": "tut-task"},
    system_message="sys", tasktrials=tasktrials, chain_type="item", hmsg="stimulus",
)
for r in runner.execute():
    print(r["trcode"], "->", r["pred_resp"].content)

----<>---- task running


t1 -> [vision_worker] [{'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAIAAACQd1PeAAAADElEQVR42mNgAAACAAFVofa1AAAAAElFTkSuQmCC', 'mime_type': 'image/png'}]


## 2. Mixed block types — routes through the planner

An image **and** text stimulus goes through the planner, which dispatches
both `vision_worker` and `text_worker` on their respective slices, then
`aggregate` combines their findings into one answer — visible here as both
workers' tagged echoes concatenated together.

In [3]:
tasktrials = {"trials": [
    {"trcode": "t2",
     "stimulus": HumanMessage(content=[image_block(RUN_DIR / "pixel.png"),
                                        {"type": "text", "text": "The participant said they felt calm during this trial."}]),
     "tasktype": "x", "parser": None, "fb": False},
]}
runner = TaskRunner(
    scanning_agent=agent, trace_cfg={"trial": "tut-", "task": "tut-task"},
    system_message="sys", tasktrials=tasktrials, chain_type="item", hmsg="stimulus",
)
for r in runner.execute():
    print(r["trcode"], "->", r["pred_resp"].content)

----<>---- task running


t2 -> [vision_worker] [{'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAIAAACQd1PeAAAADElEQVR42mNgAAACAAFVofa1AAAAAElFTkSuQmCC', 'mime_type': 'image/png'}]
[text_worker] [{'type': 'text', 'tex


## 3. Through `ScannerModel.run(custom_agent=...)`

Same seam as any `ScanningAgent` — a full simulation run over a small block
of image trials.

In [4]:
task = {
    "tasktype": "visual_search", "taskname": "supervisor_demo",
    "instructions": {"definition": ["Look at the image and describe what you see."]},
    "contexts": ["demo"], "contexts_id": ["feat"], "context_present": False,
    "chain_type": "item", "parser": "0",
    "items": {"feat": [
        {"trcode": "feat_1", "stimulus": [image_block(RUN_DIR / "pixel.png")]},
    ]},
}

card_in = psy.ExpCardInit()
card_in.proj_dir, card_in.projectname = RUN_DIR, "supervisor_demo"
card_in.task_file = task
card_in.cogtype, card_in.nsim = "no", 1
card_in.chain_type, card_in.memory = "item", "SingleTurn"

scanner = psy.ScannerModel(expcard=psy.ExpCard(card_in))
scan_results = scanner.run(custom_agent=agent)
for trial in scan_results[0]:
    print(trial["trcode"], "->", trial["pred_resp"].content)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_supervisor_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_supervisor_tutorial_run/supervisor_demo/supervisor_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

1it [00:00, 362.70it/s]


2026-07-06 11:50:32.793 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:50:32.794 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


feat_1 -> [vision_worker] [{'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAIAAACQd1PeAAAADElEQVR42mNgAAACAAFVofa1AAAAAElFTkSuQmCC', 'mime_type': 'image/png'}]


## Recap

- Supervisor, Planner, and Workers are separate LangGraph nodes: a
  single-block-type stimulus routes straight to its worker; a mixed-block
  stimulus routes through the planner, which dispatches each worker on its
  slice of the content, then an aggregate step combines their findings —
  the same shape Jockey uses for video search/text/editing workers.
- Plugs into psychscanner exactly like any other `ScanningAgent`.

---
## Further reading

Advanced applications of supervisor/planner/worker multi-agent routing:

1. **["MetaGPT: Meta Programming for A Multi-Agent Collaborative Framework"](https://arxiv.org/abs/2308.00352)** (Hong et al., 2023) — assigns standardized roles to specialized agents on an assembly line, the same supervisor→planner→workers shape used here, generalized to full software-engineering workflows.
2. **["AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation Framework"](https://arxiv.org/abs/2308.08155)** (Wu et al., 2023) — a general-purpose framework for the routing/dispatch/aggregate pattern `make_supervisor_agent` implements directly on LangGraph.